# Altmann su tre corpora appaiati: Wikipedia, Grokipedia v0.1, Grokipedia oggi

Ogni corpus viene prima analizzato **da solo**, poi i tre vengono confrontati.

Il disegno è **appaiato per titolo**: lo stesso argomento compare in tutti e tre i corpora, alla
stessa lunghezza in caratteri. È il vantaggio decisivo rispetto al confronto
"continuazione di *Guerra e pace* contro Tolstoj": qui l'argomento è controllato esattamente,
quindi le differenze osservate non possono essere attribuite al fatto che i testi parlano di
cose diverse. Di conseguenza i confronti usano test **appaiati** (Wilcoxon sui ranghi con
segno), non test fra gruppi indipendenti.

## Le domande, in ordine

1. **Dentro ciascun corpus** — le keyword auto-selezionate sono più bursty dei controlli
   appaiati in frequenza? È il test di Altmann, interno a ogni documento, indipendente da
   lunghezza e lessico.
2. **Fra corpora** — il livello di burstiness e di correlazione è lo stesso? Qui la lunghezza
   conta, e per questo tutti i testi sono troncati alla stessa `N_EFF`.
3. **L'effetto della revisione** — Grokipedia v0.1 (ottobre 2025) e la versione odierna sono
   lo stesso testo revisionato. Il processo di revisione avvicina o allontana le statistiche da
   quelle di Wikipedia? Nessun altro disegno disponibile risponde a questa domanda.

## Cosa è già stato verificato altrove, e che qui si dà per acquisito

- **Paternità.** La versione odierna è scritta da Grok in 40 casi su 40 (sovrapposizione di
  8-grammi con Wikipedia: mediana 0.0003, massimo 0.046). Nella v0.1 invece *Buddhism* è una
  copia di Wikipedia all'84%: **viene escluso da tutto il confronto**, perché in un disegno
  appaiato un titolo che cade in un corpus deve cadere in tutti.
- **Estrazione.** L'estrattore mirato di Grokipedia elimina l'artefatto degli incollamenti
  (`camelCase` per token 0.0001, identico a Wikipedia). Un'estrazione grezza dava 41 volte il
  valore di Wikipedia e avrebbe falsato ogni conteggio di parole.
- **Il riferimento letterario** (*Guerra e pace*) è incluso come ancora: è il testo su cui il
  protocollo è validato contro i numeri del paper.

## Una differenza di stile che va tenuta a mente leggendo i risultati

Grokipedia scrive frasi molto più lunghe di Wikipedia: 4.38 frasi ogni 1000 caratteri contro
6.92, cioè una frase ogni 228 caratteri contro una ogni 145. Siccome qui il tempo si conta in
caratteri, la scala su cui operano gli intervalli $\tau$ **non è la stessa** nei due corpora.
Non è un artefatto da correggere — è una proprietà reale dei testi — ma è un confonditore
possibile per qualunque differenza si trovi, e va dichiarato.

## 1. Configurazione

In [ ]:
from pathlib import Path

QUI  = Path.cwd()
BASE = QUI if (QUI / "corpora_cache").is_dir() else QUI.parent
CACHE_WIKI = QUI / "cache_wikipedia"
CACHE_GROK = QUI / "cache_grokipedia"
CORPUS_V01 = QUI / "risultati_confronto" / "corpus_v01"
DATI_CONF  = QUI / "risultati_confronto" / "dati"
CACHE_LIB  = BASE / "corpora_cache"
OUT_DIR    = QUI / "risultati_tre_corpora"

# --- lunghezza comune -------------------------------------------------------
# Il confronto ha senso solo a lunghezza uguale: sigma_tau/<tau> diverge con N.
# 60.000 e' il valore piu' alto che conserva quasi tutte le terne (il lato corto e'
# Grokipedia odierna: Moon 61.539, Albert Einstein 63.582). La cella 4 riporta
# quante terne sopravvivono a diverse soglie, cosi' la scelta e' verificabile.
N_EFF = 60_000
N_SEG_LETTERARI = 20
MIN_EVENTS = 15

# --- titoli da escludere ----------------------------------------------------
# Buddhism: la v0.1 e' copiata da Wikipedia all'84% (dati/paternita_v01.csv).
ESCLUSI = ["Buddhism"]

# --- parametri di Altmann: identici agli altri notebook ---------------------
LAG_MIN, LAG_MAX  = 4, 4000
N_LAGS            = 40
FIT_RANGE         = (100, 2000)
FIT_RANGE_STRETTO = (100, 600)          # 1% di N_EFF
N_LETTERS, N_FUNCTION, N_KEYWORDS, N_MATCHED = 19, 6, 7, 7
PROPER_CAP_RATIO = 0.6
N_NULL_REPS = 3

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
START_PHRASE  = "Well, Prince, so Genoa and Lucca"
PROMPT_CHARS  = 8262

# soglia sulla similarita' v0.1 <-> odierna per separare riscritti e invariati
SOGLIA_RISCRITTO, SOGLIA_INVARIATO = 0.50, 0.90

RANDOM_SEED = 20260814
DPI = 150
ORDINE = ["letterario", "wikipedia", "grok_v01", "grok_oggi"]
ETICHETTE = {"letterario": "Guerra e pace", "wikipedia": "Wikipedia",
             "grok_v01": "Grokipedia v0.1", "grok_oggi": "Grokipedia oggi"}
COLORI = {"letterario": "#333333", "wikipedia": "#0072B2",
          "grok_v01": "#E69F00", "grok_oggi": "#D55E00"}
print("cartella:", QUI.resolve())

In [ ]:
import re, ssl, json, math, hashlib, warnings, urllib.request
import html as htmlmod
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats as sps
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
try:
    import certifi
    HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": .25,
                     "axes.axisbelow": True, "legend.frameon": True})
warnings.filterwarnings("ignore", category=RuntimeWarning)
for d in (OUT_DIR, OUT_DIR / "figure", OUT_DIR / "dati"):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
LAGS = np.unique(np.logspace(np.log10(LAG_MIN), np.log10(LAG_MAX), N_LAGS).astype(int))
print("lag:", LAGS[:5], "...", LAGS[-2:], f"({len(LAGS)})  | scipy:", HAS_SCIPY)

## 2. Il nucleo di misura

Copiato **verbatim** dal notebook principale, dove è validato contro il paper
($\sigma_\tau/\langle\tau\rangle=0.844$ per la lettera "e" contro 0.83 di Altmann, 3.892 per
*prince* contro 3.86, $\hat\gamma_{A1}=0.98$). Non va modificato: se i notebook divergono, i
numeri smettono di essere confrontabili fra loro.

In [ ]:
WORD_CHAR = r"[^\W\d_]"
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)
SENT_END  = set(".!?")

def build_char_array(t): return np.array(list(t.lower()))
def pos_from_char(c, ch): return np.flatnonzero(c == ch)
def pos_from_set(c, s):   return np.flatnonzero(np.isin(c, list(s)))

_wcache = {}
def pos_from_word(text, w):
    if w not in _wcache:
        _wcache[w] = re.compile(r"(?<!" + WORD_CHAR + r")" + re.escape(w) +
                                r"(?!" + WORD_CHAR + r")", re.IGNORECASE | re.UNICODE)
    return np.fromiter((m.start() for m in _wcache[w].finditer(text)), dtype=np.int64)

def seme(*p):
    h = hashlib.sha256("|".join(map(str, p)).encode("utf-8")).hexdigest()
    return (int(h[:8], 16) ^ RANDOM_SEED) % (2 ** 32)

def transport_sigma2(pos, N):
    x = np.zeros(N, dtype=np.float64)
    if len(pos): x[pos[pos < N]] = 1.0
    C = np.concatenate(([0.0], np.cumsum(x)))
    out = np.full(len(LAGS), np.nan)
    for i, t in enumerate(LAGS):
        if t >= N // 4: continue
        out[i] = (C[t:] - C[:-t]).var()
    return out

def fit_gamma(s2, lo, hi, min_pts=5):
    s2 = np.asarray(s2, float)
    m = np.isfinite(s2) & (s2 > 0) & (LAGS >= lo) & (LAGS <= hi)
    if m.sum() < min_pts: return np.nan, np.nan
    X, Y = np.log10(LAGS[m].astype(float)), np.log10(s2[m])
    b, a = np.polyfit(X, Y, 1)
    r = Y - (b * X + a); sxx = ((X - X.mean()) ** 2).sum()
    se = float(np.sqrt((r ** 2).sum() / max(len(X) - 2, 1) / sxx)) if sxx > 0 else np.nan
    return float(b), se

def interevent(pos):
    return np.diff(np.asarray(pos, np.int64)) if len(pos) > 1 else np.array([], np.int64)

def burstiness_stats(tau):
    if len(tau) < 2:
        return dict(mean_tau=np.nan, sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan)
    m, s = float(tau.mean()), float(tau.std(ddof=1))
    return dict(mean_tau=m, sigma_tau=s, cv_tau=s/m if m > 0 else np.nan,
                B_goh=(s-m)/(s+m) if (s+m) > 0 else np.nan)

def null_A1(pos, N, r):
    M = len(pos)
    return np.sort(r.choice(N, size=min(M, N), replace=False)) if M else pos

def null_A2(pos, N, r):
    tau = interevent(pos)
    if len(tau) < 2: return pos
    tau = r.permutation(tau)
    new = np.concatenate(([pos[0]], pos[0] + np.cumsum(tau)))
    return new[new < N]

def analyze_sequence(pos, N, label, level, r):
    pos = np.asarray(pos, np.int64); pos = pos[pos < N]; M = len(pos)
    rec = dict(sequenza=label, livello=level, n_eventi=M, n_chars=N,
               frequenza=M/N if N else np.nan)
    if M < MIN_EVENTS:
        rec.update(dict(gamma=np.nan, gamma_se=np.nan, gamma_stretto=np.nan,
                        gamma_A1=np.nan, gamma_A2=np.nan, mean_tau=np.nan,
                        sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan, stimabile=False))
        return rec
    s2 = transport_sigma2(pos, N)
    g, se = fit_gamma(s2, *FIT_RANGE)
    gs, _ = fit_gamma(s2, *FIT_RANGE_STRETTO)
    rec.update(dict(gamma=g, gamma_se=se, gamma_stretto=gs, stimabile=True))
    rec.update(burstiness_stats(interevent(pos)))
    g1, g2 = [], []
    for _ in range(N_NULL_REPS):
        g1.append(fit_gamma(transport_sigma2(null_A1(pos, N, r), N), *FIT_RANGE)[0])
        g2.append(fit_gamma(transport_sigma2(null_A2(pos, N, r), N), *FIT_RANGE)[0])
    rec["gamma_A1"] = float(np.nanmean(g1)); rec["gamma_A2"] = float(np.nanmean(g2))
    return rec

STOPWORDS = set("""a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few for
from further had has have having he her here hers herself him himself his how i if in into is it its
itself me more most my myself no nor not of off on once only or other ought our ours ourselves out
over own same she should so some such than that the their theirs them themselves then there these
they this those through to too under until up very was we were what when where which while who whom
why with would you your yours yourself yourselves said one two would shall may might must upon""".split())
VOWELS = set("aeiou")

def word_stats(text):
    freq, cap, tot = Counter(), Counter(), Counter()
    pe, primo = 0, True
    for m in TOKEN_RE.finditer(text):
        w = m.group(0); lw = w.lower(); freq[lw] += 1
        gap = text[pe:m.start()]
        if not (primo or any(c in SENT_END for c in gap) or "\n\n" in gap):
            tot[lw] += 1
            if w[0].isupper(): cap[lw] += 1
        pe, primo = m.end(), False
    return freq, {w: (cap[w]/tot[w] if tot[w] >= 3 else 0.0) for w in freq}

def select_targets(text):
    low = text.lower()
    letters = [c for c, _ in Counter(c for c in low if c.isalpha() and c.isascii())
               .most_common(N_LETTERS)]
    freq, capr = word_stats(text)
    ordered = [w for w, _ in freq.most_common() if len(w) >= 2]
    funcs = [w for w in ordered if w in STOPWORDS][:N_FUNCTION]
    isp = lambda w: capr.get(w, 0) >= PROPER_CAP_RATIO
    isk = lambda w: isp(w) or (w not in STOPWORDS and len(w) >= 4)
    keys = [w for w in ordered if isk(w)][:N_KEYWORDS]
    ks = set(keys)
    pool = [w for w in ordered if w not in ks and not isp(w) and w not in funcs]
    matched, used = [], set()
    for k in keys[:N_MATCHED]:
        fk = freq[k]
        cand = sorted((w for w in pool if w not in used),
                      key=lambda w: (abs(math.log((freq[w]+1e-9)/(fk+1e-9))), w))
        if cand: matched.append(cand[0]); used.add(cand[0])
    return {"vc": [("vocali", "vocali", VOWELS)],
            "lettere": [("spazio", "spazio", " ")] + [(c, "lettera", c) for c in letters],
            "parole": ([(w, "funzione", w) for w in funcs] +
                       [(w, "keyword", w) for w in keys] +
                       [(w, "appaiata", w) for w in matched])}

def targets_positions(text, tg):
    carr = build_char_array(text); out = []
    for lev, items in tg.items():
        for lab, tp, key in items:
            if lev == "vc":       pos = pos_from_set(carr, key)
            elif tp == "spazio":  pos = pos_from_char(carr, " ")
            elif tp == "lettera": pos = pos_from_char(carr, key)
            else:                 pos = pos_from_word(text, key)
            out.append((lab, tp, lev, pos))
    return out

def analizza(text, meta):
    """wordset auto: i bersagli sono riselezionati dentro questo testo"""
    N = min(len(text), N_EFF); t = text[:N]
    r = np.random.default_rng(seme(meta["corpus"], meta["titolo"]))
    recs = []
    for lab, tp, lev, pos in targets_positions(t, select_targets(t)):
        rec = analyze_sequence(pos, N, lab, lev, r)
        rec.update(tipo=tp, **meta)
        recs.append(rec)
    return recs

print("nucleo di misura caricato")

## 3. Caricamento dei tre corpora

In [ ]:
APP = re.compile(r"^==+\s*(See also|References|Notes|Citations|Sources|Bibliography|"
                 r"Further reading|External links|Works cited|Footnotes|Explanatory notes|"
                 r"General sources)\s*==+\s*$", re.M | re.I)
INT = re.compile(r"^==+.*?==+\s*$", re.M)
TTS = re.compile(r'<span[^>]*data-tts-block="true"[^>]*>(.*?)</span>', re.S)

def _frag(h):
    h = re.sub(r"(?is)<(script|style|button|svg|nav|footer|aside)[^>]*>.*?</\1>", " ", h)
    h = re.sub(r"<[^>]+>", " ", h)
    h = htmlmod.unescape(h)
    h = re.sub(r"\[\d+\]", " ", h)
    h = re.sub(r"[ \t\u00a0]+", " ", h)
    return re.sub(r"\n\s*\n+", "\n\n", h).strip()

def norm(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"\n[ \t]+\n", "\n\n", t)
    return re.sub(r"\n{3,}", "\n\n", t).strip()

def safe(t): return re.sub(r"[^A-Za-z0-9_]", "", t.replace(" ", "_"))[:60]

def carica_wikipedia(t):
    fn = CACHE_WIKI / (hashlib.sha256(t.encode()).hexdigest()[:14] + ".txt")
    if not fn.exists(): return None
    tx = fn.read_bytes().decode("utf-8", errors="replace")
    m = APP.search(tx)
    if m: tx = tx[:m.start()]
    return norm(INT.sub("", tx))

def carica_grok_oggi(t):
    fn = CACHE_GROK / (safe(t) + ".html")
    if not fn.exists(): return None
    p = fn.read_bytes().decode("utf-8", errors="replace")
    b = [_frag(x) for x in TTS.findall(p)]
    b = [x for x in b if len(x) > 40]
    return norm("\n\n".join(b)) if b else None

def carica_grok_v01(t):
    fn = CORPUS_V01 / (safe(t) + ".txt")
    return norm(fn.read_bytes().decode("utf-8", errors="replace")) if fn.exists() else None

RIEP = pd.read_csv(DATI_CONF / "riepilogo.csv")
TITOLI = [t for t in RIEP["titolo"] if t not in ESCLUSI]
print(f"titoli dal confronto precedente: {len(RIEP)} | esclusi {ESCLUSI} -> {len(TITOLI)}")

TESTI = {}
mancanti = []
for t in TITOLI:
    d = {"wikipedia": carica_wikipedia(t), "grok_v01": carica_grok_v01(t),
         "grok_oggi": carica_grok_oggi(t)}
    if any(v is None for v in d.values()):
        mancanti.append((t, [k for k, v in d.items() if v is None])); continue
    TESTI[t] = d
print(f"titoli con tutti e tre i corpora: {len(TESTI)}")
if mancanti:
    for t, k in mancanti: print(f"  mancante: {t} -> {k}")

LUN = pd.DataFrame({c: {t: len(v[c]) for t, v in TESTI.items()}
                    for c in ["wikipedia", "grok_v01", "grok_oggi"]})
print("\nlunghezze (caratteri):")
print(LUN.describe().loc[["min", "50%", "max"]].round(0).to_string())

## 4. Lunghezza comune

Una terna è utilizzabile solo se **tutti e tre** i testi arrivano a `N_EFF`. Il lato corto è
Grokipedia odierna. La tabella mostra quante terne sopravvivono a diverse soglie, così la scelta
di `N_EFF` è verificabile invece che asserita.

In [ ]:
print("terne complete al variare della soglia:\n")
print(f"  {'N_EFF':>8s}  {'terne':>6s}  titoli persi")
for s in [40_000, 50_000, 60_000, 70_000, 80_000, 90_000]:
    ok = (LUN >= s).all(axis=1)
    persi = list(LUN.index[~ok])
    marca = "  <-- scelto" if s == N_EFF else ""
    print(f"  {s:>8,}  {int(ok.sum()):>6d}  "
          f"{(', '.join(persi[:4]) + (' ...' if len(persi) > 4 else '')) if persi else '-'}{marca}")

USABILI = sorted(LUN.index[(LUN >= N_EFF).all(axis=1)])
print(f"\nN_EFF = {N_EFF:,} -> {len(USABILI)} terne usabili")

# riferimento letterario alla stessa lunghezza
GS = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
GE = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
ST = re.compile(r"^[ \t]*(?:BOOK\s+[A-Z]+[^\n]*|CHAPTER\s+[IVXLCDM\d]+[^\n]*|"
                r"(?:FIRST|SECOND)\s+EPILOGUE[^\n]*|EPILOGUE[^\n]*|CONTENTS[^\n]*|"
                r"PART\s+[IVXLCDM\d]+[^\n]*|APPENDIX[^\n]*|\d+)[ \t]*$", re.M)
def ctx_ssl():
    if HAS_CERTIFI:
        try: return ssl.create_default_context(cafile=certifi.where())
        except Exception: pass
    return ssl.create_default_context()
fn = CACHE_LIB / (hashlib.sha256(GUTENBERG_URL.encode()).hexdigest()[:12] + ".txt")
if fn.exists():
    raw = fn.read_bytes().decode("utf-8", errors="replace")
else:
    req = urllib.request.Request(GUTENBERG_URL, headers={"User-Agent": "research/1.0"})
    with urllib.request.urlopen(req, timeout=90, context=ctx_ssl()) as r:
        raw = r.read().decode("utf-8", errors="replace")
    CACHE_LIB.mkdir(parents=True, exist_ok=True); fn.write_bytes(raw.encode("utf-8"))
raw = raw.replace("\r\n", "\n").replace("\r", "\n")
raw = raw[GS.search(raw).end():]; raw = raw[:GE.search(raw).start()]
raw = ST.sub("", raw[raw.find(START_PHRASE):])
raw = re.sub(r"\n[ \t]+\n", "\n\n", raw)
BODY = re.sub(r"\n{3,}", "\n\n", raw).strip()[PROMPT_CHARS:]
SEG = [BODY[s:s+N_EFF] for s in
       np.linspace(0, max(len(BODY)-N_EFF, 0), N_SEG_LETTERARI).astype(int)]
print(f"riferimento letterario: {len(SEG)} segmenti da {N_EFF:,}")

## 5. Analisi — prima ogni corpus da solo

In [ ]:
REC = []
for c in ["wikipedia", "grok_v01", "grok_oggi"]:
    print(f"{ETICHETTE[c]:18s}", end=" ", flush=True)
    for t in USABILI:
        REC += analizza(TESTI[t][c], dict(corpus=c, titolo=t))
        print(".", end="", flush=True)
    print(" fatto")
print(f"{ETICHETTE['letterario']:18s}", end=" ", flush=True)
for i, s in enumerate(SEG):
    REC += analizza(s, dict(corpus="letterario", titolo=f"wrnpc_{i:02d}"))
    print(".", end="", flush=True)
print(" fatto")

A = pd.DataFrame(REC)
A.to_csv(OUT_DIR / "dati" / "sequenze.csv", index=False)
print(f"\nrecord: {len(A):,}")
print(A.groupby("corpus")["stimabile"].agg(["size", "mean"]).round(3).to_string())
print("\ncontrollo del null model A1 (atteso ~1.00 ovunque):")
print(A[A["stimabile"]].pivot_table(index="corpus", columns="livello",
                                    values="gamma_A1").round(3).to_string())

In [ ]:
# --- risultati per corpus, livello per livello ---
LIV = [("lettera", "lettere"), ("funzione", "parole funzione"),
       ("appaiata", "controlli appaiati"), ("keyword", "keyword")]

def per_doc(c, tipo, col):
    s = A[(A["corpus"] == c) & (A["tipo"] == tipo)]
    return s.groupby("titolo")[col].mean().dropna()

righe = []
for c in ORDINE:
    for tipo, nome in LIV:
        cv, ga = per_doc(c, tipo, "cv_tau"), per_doc(c, tipo, "gamma")
        righe.append(dict(corpus=c, tipo=tipo, n_doc=len(cv),
                          cv=cv.mean(), cv_sd=cv.std(),
                          gamma=ga.mean(), gamma_sd=ga.std()))
LEV = pd.DataFrame(righe)
LEV.to_csv(OUT_DIR / "dati" / "livelli.csv", index=False)

for c in ORDINE:
    s = LEV[LEV["corpus"] == c]
    print(f"\n{ETICHETTE[c]}  ({int(s['n_doc'].max())} testi, N = {N_EFF:,} caratteri)")
    print(f"  {'livello':20s} {'sigma_tau/<tau>':>18s} {'gamma':>16s}")
    for tipo, nome in LIV:
        x = s[s["tipo"] == tipo].iloc[0]
        print(f"  {nome:20s} {x['cv']:>10.2f} +-{x['cv_sd']:5.2f} "
              f"{x['gamma']:>9.3f} +-{x['gamma_sd']:5.3f}")

print("\n\nRicorrenza delle keyword auto (una parola bursty deve prima ricorrere):")
for c in ORDINE:
    k = A[(A["corpus"] == c) & (A["tipo"] == "keyword")]
    top = k.groupby("titolo")["n_eventi"].max()
    print(f"  {ETICHETTE[c]:18s} piu' ricorrente: mediana {top.median():5.0f} "
          f"[{top.min():4.0f}-{top.max():5.0f}] | occorrenze mediane "
          f"{k['n_eventi'].median():4.0f} | stimabili {100*(k['n_eventi']>=MIN_EVENTS).mean():3.0f}%")

## 6. Domanda 1 — l'effetto di Altmann esiste dentro ogni corpus?

Appaiamento **posizionale** dentro ogni testo: `select_targets` costruisce `matched[i]` come il
controllo a frequenza più vicina a `keys[i]`, e i record sono accodati in quell'ordine. Non si
filtra su `stimabile` prima di appaiare, altrimenti le coppie si sfasano e il test misura
rumore.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z**2/n
    c = (p + z**2/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))/d
    return (max(c-h, 0.0), min(c+h, 1.0))

def test_interno(sub, col):
    v = tot = 0; diffs = []
    for _, doc in sub.groupby("titolo"):
        k = doc[doc["tipo"] == "keyword"].sort_index()
        m = doc[doc["tipo"] == "appaiata"].sort_index()
        for ik, im in zip(k.index, m.index):
            a, b = A.at[ik, col], A.at[im, col]
            if np.isfinite(a) and np.isfinite(b):
                tot += 1; v += int(a > b); diffs.append(a - b)
    lo, hi = wilson(v, tot)
    p = float(sps.binomtest(v, tot, 0.5).pvalue) if (HAS_SCIPY and tot) else np.nan
    return dict(vittorie=v, confronti=tot, frazione=v/tot if tot else np.nan,
                ic_lo=lo, ic_hi=hi, p=p,
                diff_media=float(np.mean(diffs)) if diffs else np.nan)

righe = []
for c in ORDINE:
    sub = A[A["corpus"] == c]
    for col in ["cv_tau", "gamma"]:
        r = test_interno(sub, col)
        r.update(corpus=c, metrica=col, documenti=sub["titolo"].nunique())
        righe.append(r)
PT = pd.DataFrame(righe)[["corpus", "metrica", "documenti", "vittorie", "confronti",
                          "frazione", "ic_lo", "ic_hi", "p", "diff_media"]]
PT.to_csv(OUT_DIR / "dati" / "test_interno.csv", index=False)

print("Le keyword battono il proprio controllo a frequenza appaiata?")
print("(0.5 = nessun effetto; l'IC e' di Wilson)\n")
for met, nome in [("cv_tau", "BURSTINESS  sigma_tau/<tau>"), ("gamma", "CORRELAZIONE  gamma")]:
    print(nome)
    for c in ORDINE:
        x = PT[(PT["corpus"] == c) & (PT["metrica"] == met)].iloc[0]
        st = "" if not np.isfinite(x["p"]) else (
            " ***" if x["p"] < 1e-3 else " **" if x["p"] < 1e-2
            else " *" if x["p"] < .05 else "  n.s.")
        esito = "SI" if x["ic_lo"] > 0.5 else "no"
        print(f"  {ETICHETTE[c]:18s} {int(x['vittorie']):4d}/{int(x['confronti']):4d} = "
              f"{x['frazione']:.2f} [{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]  "
              f"p={x['p']:.1e}{st:7s} effetto: {esito}")
    print()

## 7. Domanda 2 — confronto fra corpora, appaiato per titolo

Ogni titolo compare in tutti e tre i corpora, quindi il confronto è **appaiato**: si usa il test
di Wilcoxon sui ranghi con segno sulle differenze per titolo, non un test fra gruppi
indipendenti. È molto più potente, e soprattutto elimina la variabilità fra argomenti.

Il riferimento letterario non è appaiato (sono segmenti di un romanzo, non gli stessi
argomenti): per quello si riporta solo la media, senza test appaiato.

In [ ]:
def vettore(c, tipo, col):
    return per_doc(c, tipo, col).reindex(USABILI)

def confronta(c1, c2, tipo, col):
    a, b = vettore(c1, tipo, col), vettore(c2, tipo, col)
    m = a.notna() & b.notna()
    a, b = a[m], b[m]
    if len(a) < 5:
        return dict(n=len(a), mediana_diff=np.nan, p=np.nan, vince_c1=np.nan)
    p = float(sps.wilcoxon(a, b).pvalue) if HAS_SCIPY else np.nan
    return dict(n=len(a), mediana_diff=float(np.median(a - b)), p=p,
                vince_c1=float((a > b).mean()))

COPPIE = [("grok_oggi", "wikipedia"), ("grok_v01", "wikipedia"), ("grok_oggi", "grok_v01")]
righe = []
for c1, c2 in COPPIE:
    for tipo, nome in LIV:
        for col in ["cv_tau", "gamma"]:
            r = confronta(c1, c2, tipo, col)
            r.update(corpus_1=c1, corpus_2=c2, tipo=tipo, metrica=col)
            righe.append(r)
CONF = pd.DataFrame(righe)
CONF.to_csv(OUT_DIR / "dati" / "confronti_appaiati.csv", index=False)

for col, nome in [("cv_tau", "BURSTINESS  sigma_tau/<tau>"), ("gamma", "CORRELAZIONE  gamma")]:
    print(nome)
    print(f"  {'confronto':34s} {'livello':20s} {'n':>3s} {'diff mediana':>13s} "
          f"{'vince 1o':>9s} {'p':>9s}")
    for c1, c2 in COPPIE:
        for tipo, nl in LIV:
            x = CONF[(CONF.corpus_1 == c1) & (CONF.corpus_2 == c2) &
                     (CONF.tipo == tipo) & (CONF.metrica == col)].iloc[0]
            st = "" if not np.isfinite(x["p"]) else (
                "***" if x["p"] < 1e-3 else "**" if x["p"] < 1e-2
                else "*" if x["p"] < .05 else "")
            print(f"  {ETICHETTE[c1]+' vs '+ETICHETTE[c2]:34s} {nl:20s} {int(x['n']):>3d} "
                  f"{x['mediana_diff']:>+13.3f} {x['vince_c1']:>9.0%} {x['p']:>9.1e} {st}")
    print()

print("Medie per corpus (per riferimento, incluso il letterario non appaiato):")
print(f"  {'livello':20s} " + " ".join(f"{ETICHETTE[c][:15]:>16s}" for c in ORDINE))
for col, nome in [("cv_tau", "sigma_tau/<tau>"), ("gamma", "gamma")]:
    print(f"  --- {nome} ---")
    for tipo, nl in LIV:
        vals = []
        for c in ORDINE:
            x = LEV[(LEV.corpus == c) & (LEV.tipo == tipo)].iloc[0]
            vals.append(f"{x[col if col=='gamma' else 'cv']:>9.3f}+-{x[col.replace('cv_tau','cv')+'_sd' if col=='cv_tau' else 'gamma_sd']:5.3f}"
                        if col == "gamma" else f"{x['cv']:>9.2f}+-{x['cv_sd']:5.2f}")
        print(f"  {nl:20s} " + " ".join(f"{v:>16s}" for v in vals))

## 8. Domanda 3 — l'effetto della revisione

Il confronto v0.1 → oggi è fra due versioni **dello stesso articolo**, quindi la differenza è
attribuibile al processo di revisione e non al contenuto. Si separano gli articoli riscritti a
fondo da quelli rimasti quasi invariati: se la revisione ha un effetto sulle statistiche, deve
vedersi soprattutto nel primo gruppo.

In [ ]:
W = pd.read_csv(DATI_CONF / "revisioni_wayback.csv")
sim = W.set_index("titolo")["sim_v01_attuale"].reindex(USABILI)
gruppo = pd.Series(np.where(sim < SOGLIA_RISCRITTO, "riscritto",
                   np.where(sim > SOGLIA_INVARIATO, "invariato", "intermedio")),
                   index=USABILI)
print("articoli per gruppo di revisione:")
print(gruppo.value_counts().to_string())
print(f"\n(riscritto = similarita' v0.1<->oggi < {SOGLIA_RISCRITTO}, "
      f"invariato = > {SOGLIA_INVARIATO})\n")

righe = []
for g in ["riscritto", "intermedio", "invariato"]:
    tit = [t for t in USABILI if gruppo[t] == g]
    if len(tit) < 3: continue
    for tipo, nl in LIV:
        for col in ["cv_tau", "gamma"]:
            a = vettore("grok_oggi", tipo, col).reindex(tit)
            b = vettore("grok_v01", tipo, col).reindex(tit)
            w = vettore("wikipedia", tipo, col).reindex(tit)
            m = a.notna() & b.notna() & w.notna()
            if m.sum() < 3: continue
            p = float(sps.wilcoxon(a[m], b[m]).pvalue) if (HAS_SCIPY and m.sum() >= 5) else np.nan
            # la revisione avvicina a Wikipedia?
            d_prima = (b[m] - w[m]).abs().median()
            d_dopo  = (a[m] - w[m]).abs().median()
            righe.append(dict(gruppo=g, n=int(m.sum()), tipo=tipo, metrica=col,
                              v01=b[m].median(), oggi=a[m].median(), wiki=w[m].median(),
                              delta_oggi_v01=float(np.median(a[m]-b[m])), p=p,
                              dist_wiki_v01=d_prima, dist_wiki_oggi=d_dopo,
                              avvicina=d_dopo < d_prima))
REV = pd.DataFrame(righe)
REV.to_csv(OUT_DIR / "dati" / "effetto_revisione.csv", index=False)

for col, nome in [("cv_tau", "BURSTINESS"), ("gamma", "CORRELAZIONE gamma")]:
    print(f"{nome}: la revisione avvicina i valori a Wikipedia?")
    print(f"  {'gruppo':12s} {'livello':20s} {'n':>3s} {'v0.1':>8s} {'oggi':>8s} "
          f"{'wiki':>8s} {'|d| prima':>10s} {'|d| dopo':>9s}  esito")
    for _, x in REV[REV["metrica"] == col].iterrows():
        nl = dict(LIV)[x["tipo"]]
        print(f"  {x['gruppo']:12s} {nl:20s} {int(x['n']):>3d} {x['v01']:>8.3f} "
              f"{x['oggi']:>8.3f} {x['wiki']:>8.3f} {x['dist_wiki_v01']:>10.3f} "
              f"{x['dist_wiki_oggi']:>9.3f}  {'avvicina' if x['avvicina'] else 'allontana'}")
    print()

## 9. Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.9))

ax = axes[0]
sub = PT[PT["metrica"] == "cv_tau"].set_index("corpus").reindex(ORDINE)
y = np.arange(len(ORDINE))
err = np.clip(np.vstack([sub["frazione"]-sub["ic_lo"], sub["ic_hi"]-sub["frazione"]]), 0, None)
for i, c in enumerate(ORDINE):
    ax.errorbar([sub.loc[c, "frazione"]], [i], xerr=err[:, i:i+1], fmt="o", ms=10,
                capsize=5, color=COLORI[c], lw=2)
ax.set_yticks(y); ax.set_yticklabels([ETICHETTE[c] for c in ORDINE], fontsize=9)
ax.set_ylim(-.6, len(ORDINE)-.4); ax.set_xlim(0, 1)
ax.axvline(.5, color="grey", ls=":", lw=1.4)
ax.set_xlabel("frazione di coppie con keyword più bursty")
ax.set_title("A) l'effetto di Altmann esiste?\n(0.5 = nessun effetto)", fontsize=10)

ax = axes[1]
tipi = [t for t, _ in LIV]; xs = np.arange(len(tipi))
for i, c in enumerate(ORDINE):
    s = LEV[LEV["corpus"] == c].set_index("tipo").reindex(tipi)
    ax.errorbar(xs + .07*(i-1.5), s["cv"], yerr=s["cv_sd"], marker="osD^"[i], ms=7,
                lw=1.8, capsize=3.5, color=COLORI[c], label=ETICHETTE[c])
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
ax.set_title("B) burstiness per livello linguistico", fontsize=10)
ax.legend(fontsize=7.5)

ax = axes[2]
dati, et, cc = [], [], []
for c in ORDINE:
    v = per_doc(c, "keyword", "cv_tau")
    if len(v): dati.append(v.values); et.append(f"{ETICHETTE[c]}\n({len(v)})"); cc.append(c)
bp = ax.boxplot(dati, tick_labels=et, showmeans=True, widths=.55, patch_artist=True)
for b, c in zip(bp["boxes"], cc):
    b.set(facecolor=COLORI[c], alpha=.30)
for i, v in enumerate(dati):
    ax.scatter(np.full(len(v), i+1) + rng.normal(0, .05, len(v)), v, s=14,
               color="k", alpha=.45, zorder=4)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$ delle keyword")
ax.set_title("C) burstiness delle keyword per testo", fontsize=10)
ax.tick_params(axis="x", labelsize=7)
fig.suptitle(f"Altmann su tre corpora appaiati per titolo  "
             f"({len(USABILI)} terne, N = {N_EFF:,} caratteri)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "tre_corpora.png"); plt.show()

# --- figura appaiata: ogni titolo, v0.1 -> oggi, contro Wikipedia ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, (tipo, nl) in zip(axes, [("keyword", "keyword"), ("lettera", "lettere")]):
    w = vettore("wikipedia", tipo, "cv_tau")
    a = vettore("grok_v01", tipo, "cv_tau")
    b = vettore("grok_oggi", tipo, "cv_tau")
    m = w.notna() & a.notna() & b.notna()
    for t in np.array(USABILI)[m.values]:
        ax.plot([0, 1], [a[t], b[t]], color="grey", lw=.8, alpha=.5, zorder=1)
    ax.scatter(np.zeros(m.sum()), a[m], s=34, color=COLORI["grok_v01"],
               label="Grokipedia v0.1", zorder=3, edgecolors="k", linewidths=.4)
    ax.scatter(np.ones(m.sum()), b[m], s=34, color=COLORI["grok_oggi"],
               label="Grokipedia oggi", zorder=3, edgecolors="k", linewidths=.4)
    ax.axhline(w[m].median(), color=COLORI["wikipedia"], lw=2.2, label="Wikipedia (mediana)")
    ax.axhline(LEV[(LEV.corpus=="letterario") & (LEV.tipo==tipo)]["cv"].iloc[0],
               color="#333333", lw=1.6, ls="--", label="Guerra e pace")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["v0.1", "oggi"])
    ax.set_xlim(-.35, 1.35)
    ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
    ax.set_title(f"{nl}: effetto della revisione, articolo per articolo", fontsize=10)
    ax.legend(fontsize=7.5)
fig.suptitle("La revisione di Grokipedia sposta le statistiche verso Wikipedia?",
             fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "effetto_revisione.png"); plt.show()

## 10. Sintesi

## Decomposizione dell'Eq. 4: da dove viene la correlazione a lungo raggio

Altmann et al. [1] mostrano che, legando lo spettro di potenza a frequenza nulla alla
statistica degli intervalli,

$$S(0)=\frac{\sigma_\tau^2}{\langle\tau\rangle^3}\left(1+2\sum_k C_\tau(k)\right)$$

la divergenza $S(0)\to\infty$ che caratterizza le correlazioni a lungo raggio può avere
**due origini distinte**: la *burstiness*, cioè la coda larga di $p(\tau)$ con $\sigma_\tau$
divergente, oppure le *correlazioni fra gli intervalli*, cioè $C_\tau(k)$ non sommabile.

I due null model le separano sperimentalmente. **A1** rimescola gli $\{0,1\}$ e distrugge
entrambe: è il controllo, deve dare $\hat\gamma_{A1}\simeq 1$. **A2** permuta gli intervalli
$\tau_i$, quindi **conserva esattamente $p(\tau)$ e distrugge $C_\tau(k)$**: ciò che
sopravvive in $\hat\gamma_{A2}$ è il contributo della sola burstiness.

Si definisce quindi la quota di burstiness

$$q=\frac{\hat\gamma_{A2}-1}{\hat\gamma-1}$$

che vale $\simeq 1$ se la correlazione viene tutta dalla coda di $p(\tau)$ e $\simeq 0$ se
viene tutta dall'ordine degli intervalli. È l'indicatore che rende quantitativa la Fig. 2 del
paper, dove la lettera "e" risulta correlata **ma non** bursty e la parola *prince* entrambe.

Il rapporto ha denominatore $\hat\gamma-1$, che è piccolo nei corpora meno correlati: si
calcola perciò **per documento**, escludendo i casi con denominatore sotto 0.05, e si riporta
la mediana con i quartili invece della media.

In [ ]:
# --- gamma e' significativamente > 1? (correlazione a lungo raggio o diffusione normale) ---
print("Test dei valori per documento contro gamma = 1. gamma_A1 e' il controllo.\n")
print(f"  {'corpus':17s}{'livello':20s}{'gamma':>16s}{'p vs 1':>10s}{'gamma_A1':>10s}")
righe = []
for c in ORDINE:
    for tipo, nome in LIV:
        s = A[(A.corpus == c) & (A.tipo == tipo)]
        g = s.groupby("titolo")["gamma"].mean().dropna()
        a1 = s.groupby("titolo")["gamma_A1"].mean().dropna()
        if len(g) < 5: continue
        p = float(sps.ttest_1samp(g, 1.0).pvalue) if HAS_SCIPY else np.nan
        st = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < .05 else "n.s."
        print(f"  {ETICHETTE[c]:17s}{nome:20s}{g.mean():>9.3f}+-{g.std():<5.3f}"
              f"{p:>10.1e}{a1.mean():>10.3f}  {st}")
        righe.append(dict(corpus=c, tipo=tipo, gamma=g.mean(), gamma_sd=g.std(),
                          p_vs_1=p, gamma_A1=a1.mean(), n_doc=len(g)))
    print()
LR = pd.DataFrame(righe)
LR.to_csv(OUT_DIR / "dati" / "lungo_raggio.csv", index=False)
print(f"celle con gamma > 1 significativo (p < 0.05): "
      f"{int((LR['p_vs_1'] < 0.05).sum())}/{len(LR)}")
print(f"gamma_A1 medio su tutte le celle: {LR['gamma_A1'].mean():.3f}  (atteso 1.00)")

In [ ]:
# --- la quota di burstiness q = (gamma_A2 - 1)/(gamma - 1) ---
SOGLIA_DEN = 0.05      # sotto questo gamma-1 il rapporto e' instabile

def quota_per_doc(c, tipo):
    s = A[(A.corpus == c) & (A.tipo == tipo)]
    d = s.groupby("titolo")[["gamma", "gamma_A2"]].mean().dropna()
    d = d[d["gamma"] - 1 > SOGLIA_DEN]
    return ((d["gamma_A2"] - 1) / (d["gamma"] - 1)).clip(-0.5, 1.5)

print("Quota di burstiness q (mediana [quartili]).  q~1 = tutta burstiness, "
      "q~0 = tutta C_tau(k)\n")
print(f"  {'corpus':17s}{'livello':20s}{'gamma':>8s}{'gamma_A2':>10s}{'q':>22s}{'n doc':>7s}")
righe = []
for c in ORDINE:
    for tipo, nome in LIV:
        s = A[(A.corpus == c) & (A.tipo == tipo)]
        d = s.groupby("titolo")[["gamma", "gamma_A2"]].mean().dropna()
        if len(d) < 5: continue
        q = quota_per_doc(c, tipo)
        if len(q) >= 5:
            med = float(q.median()); lo, hi = np.percentile(q, [25, 75])
            qs = f"{med:>8.2f} [{lo:+.2f},{hi:+.2f}]"
        else:
            med = lo = hi = np.nan; qs = f"{'n.d.':>22s}"
        print(f"  {ETICHETTE[c]:17s}{nome:20s}{d['gamma'].mean():>8.3f}"
              f"{d['gamma_A2'].mean():>10.3f}{qs:>22s}{len(q):>7d}")
        righe.append(dict(corpus=c, tipo=tipo, gamma=d["gamma"].mean(),
                          gamma_A2=d["gamma_A2"].mean(), q=med, q_lo=lo, q_hi=hi, n=len(q)))
    print()
Q = pd.DataFrame(righe)
Q.to_csv(OUT_DIR / "dati" / "quota_burstiness.csv", index=False)

print("La dissociazione della Fig. 2 di Altmann, sui quattro corpora:")
print(f"  {'corpus':17s}{'q lettere':>12s}{'q keyword':>12s}{'contrasto':>12s}")
for c in ORDINE:
    l = Q[(Q.corpus == c) & (Q.tipo == "lettera")]["q"]
    k = Q[(Q.corpus == c) & (Q.tipo == "keyword")]["q"]
    if not len(l) or not len(k) or not np.isfinite(l.iloc[0]): continue
    print(f"  {ETICHETTE[c]:17s}{l.iloc[0]:>12.2f}{k.iloc[0]:>12.2f}"
          f"{k.iloc[0]-l.iloc[0]:>+12.2f}")
print("\n  Nelle lettere q ~ 0: correlate ma NON bursty, la correlazione viene da C_tau(k).")
print("  Nelle keyword q e' alta: la correlazione viene dalla coda di p(tau).")

print("\nConfronto appaiato per titolo sulla quota:")
for tipo, nome in [("keyword", "keyword"), ("lettera", "lettere")]:
    w, g = quota_per_doc("wikipedia", tipo), quota_per_doc("grok_oggi", tipo)
    i = w.index.intersection(g.index)
    if len(i) < 6: continue
    p = float(sps.wilcoxon(w[i], g[i]).pvalue) if HAS_SCIPY else np.nan
    print(f"  {nome:9s} n={len(i):3d}   Wikipedia {w[i].median():+.2f}   "
          f"Grokipedia {g[i].median():+.2f}   diff mediana {np.median(w[i]-g[i]):+.2f}"
          f"   p = {p:.3f}")

In [ ]:
# --- figura: la decomposizione ---
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
xs = np.arange(len(LIV))

ax = axes[0]
for i, c in enumerate(ORDINE):
    s = Q[Q.corpus == c].set_index("tipo").reindex([t for t, _ in LIV])
    err = np.vstack([np.clip(s["q"] - s["q_lo"], 0, None),
                     np.clip(s["q_hi"] - s["q"], 0, None)])
    ax.errorbar(xs + .06*(i-1.5), s["q"], yerr=err, marker="os^D"[i], ms=8, lw=1.8,
                capsize=3, color=COLORI[c], label=ETICHETTE[c])
ax.axhline(0, color="grey", ls=":", lw=1.2)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.annotate("tutta burstiness", (len(LIV)-.6, 1.0), fontsize=8, va="bottom", ha="right",
            color="grey")
ax.annotate("tutta $C_\\tau(k)$", (len(LIV)-.6, 0.0), fontsize=8, va="bottom", ha="right",
            color="grey")
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel("quota di burstiness  $q$")
ax.set_title("A) origine della correlazione a lungo raggio\n"
             "$q=(\\hat\\gamma_{A2}-1)/(\\hat\\gamma-1)$", fontsize=10)
ax.legend(fontsize=7)

ax = axes[1]
for i, c in enumerate(ORDINE):
    s = Q[Q.corpus == c].set_index("tipo").reindex([t for t, _ in LIV])
    ax.plot(xs, s["gamma"], "-", marker="os^D"[i], ms=7, lw=1.8, color=COLORI[c],
            label=ETICHETTE[c])
    ax.plot(xs, s["gamma_A2"], "--", marker="os^D"[i], ms=5, lw=1.2, color=COLORI[c],
            alpha=.55)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel("$\\hat\\gamma$")
ax.set_title("B) $\\hat\\gamma$ (pieno) e $\\hat\\gamma_{A2}$ (tratteggio)\n"
             "la distanza fra le due e' il contributo di $C_\\tau(k)$", fontsize=10)
ax.legend(fontsize=7)

fig.suptitle("Decomposizione dell'Eq. 4 di Altmann\n"
             "tre corpora enciclopedici piu' l'ancora letteraria  "
             f"(N = {N_EFF:,} caratteri, {len(TITOLI)} terne)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "decomposizione_eq4.png"); plt.show()

## Analoghi diretti delle Fig. 2 e 3 di Altmann et al.

Le figure che seguono replicano sui quattro corpora le due figure centrali del paper. Sono
costruite **solo** su testi che superano i controlli di qualità: i testi generati dai modelli
Qwen, analizzati in una fase precedente del lavoro, sono esclusi perché 239 su 242 risultano
degenerati (loop di ripetizione o collasso dell'alfabeto), e includerli produrrebbe pannelli
che mostrano rumore.

**Fig. 2 del paper** — per una lettera e per un sostantivo si mostrano la distribuzione
cumulativa degli intervalli $P(>\tau)$ e la curva $\sigma_X^2(t)$ con i due null model. Il
risultato atteso è la dissociazione: la lettera ha $p(\tau)$ a decadimento rapido e
$\hat\gamma_{A2}\simeq1$ (correlata ma non bursty), il sostantivo ha coda larga e
$\hat\gamma_{A2}\simeq\hat\gamma$ (la correlazione viene dalla burstiness).

**Fig. 3 del paper** — il diagramma burstiness-correlazione, con le lettere in basso a
sinistra e i sostantivi in alto a destra.

In [ ]:
# --- curve complete per un documento rappresentativo di ciascun corpus ---
def transport_A(pos, N, r, n_null=3):
    """sigma^2_X(t) per originale e per i due null model, piu' i tau"""
    s2 = transport_sigma2(pos, N)
    a1 = np.zeros_like(s2); a2 = np.zeros_like(s2)
    for _ in range(n_null):
        a1 += transport_sigma2(null_A1(pos, N, r), N) / n_null
        a2 += transport_sigma2(null_A2(pos, N, r), N) / n_null
    return dict(s2=s2, s2_A1=a1, s2_A2=a2, tau=np.diff(pos))

def rappresentativo(corpus):
    """il documento con la keyword auto piu' ricorrente: e' il caso piu' leggibile"""
    s = A[(A.corpus == corpus) & (A.tipo == "keyword")]
    if not len(s): return None
    return s.loc[s["n_eventi"].idxmax(), "titolo"]

# accesso ai testi: TESTI e' indicizzato per titolo, il letterario e' la lista SEG
def testo_di(corpus, lab):
    if corpus == "letterario":
        return SEG[int(lab.split("_")[1])]
    return TESTI[lab][corpus]

def documenti_di(corpus, n=None):
    if corpus == "letterario":
        v = [(f"wrnpc_{i:02d}", SEG[i]) for i in range(len(SEG))]
    else:
        v = [(t, TESTI[t][corpus]) for t in USABILI]
    return v[:n] if n else v

CURVE = {}
for c in ORDINE:
    lab = rappresentativo(c)
    if lab is None: continue
    testo = testo_di(c, lab)
    t = testo[:N_EFF]
    carr = np.array(list(t.lower()))
    r = np.random.default_rng(seme(c, lab, "curve"))
    # lettera piu' frequente e keyword piu' ricorrente di quel documento
    d = A[(A.corpus == c) & (A.titolo == lab)]
    kw = d[d.tipo == "keyword"].sort_values("n_eventi", ascending=False)["sequenza"].iloc[0]
    let = d[d.tipo == "lettera"].sort_values("n_eventi", ascending=False)["sequenza"].iloc[0]
    CURVE[c] = dict(label=lab, lettera=let, keyword=kw,
                    L=transport_A(np.flatnonzero(carr == let), N_EFF, r),
                    K=transport_A(pos_from_word(t, kw), N_EFF, r))
    print(f"  {ETICHETTE[c]:17s} {lab[:28]:28s} lettera '{let}'  keyword '{kw}'")

In [ ]:
# --- analogo della Fig. 2 di Altmann et al., sui quattro corpora ---
def cum_dist(tau):
    if len(tau) < 2: return np.array([]), np.array([])
    s = np.sort(tau)
    return s, 1.0 - np.arange(len(s))/len(s)

def loglog(ax, x, y, **kw):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if m.sum() >= 2: ax.plot(x[m], y[m], **kw)

righe = [c for c in ORDINE if c in CURVE]
fig, axes = plt.subplots(len(righe), 4, figsize=(16.5, 3.5*len(righe)), squeeze=False)
for i, c in enumerate(righe):
    cur = CURVE[c]
    for j, (chiave, nome) in enumerate([("L", f'lettera "{cur["lettera"]}"'),
                                        ("K", f'parola "{cur["keyword"]}"')]):
        d = cur[chiave]
        ax = axes[i][2*j]
        x, y = cum_dist(d["tau"])
        loglog(ax, x, y, color="k", lw=1.8)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(r"$\tau$"); ax.set_ylabel(r"$P(>\tau)$")
        cv = d["tau"].std(ddof=1)/d["tau"].mean() if len(d["tau"]) > 1 else np.nan
        ax.set_title(f"{ETICHETTE[c]} — {nome}\n" + r"$\sigma_\tau/\langle\tau\rangle$ = "
                     + f"{cv:.2f}", fontsize=9)
        ax = axes[i][2*j+1]
        for arr, kw in [(d["s2"], dict(color="k", lw=1.8, marker="o", ms=2.5, label="originale")),
                        (d["s2_A1"], dict(color="#0072B2", lw=1.2, ls="--", label="A1")),
                        (d["s2_A2"], dict(color="#D55E00", lw=1.2, ls="-.", label="A2"))]:
            loglog(ax, LAGS, arr, **kw)
        g = fit_gamma(d["s2"], *FIT_RANGE)[0]
        g2 = fit_gamma(d["s2_A2"], *FIT_RANGE)[0]
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.axvspan(*FIT_RANGE, color="grey", alpha=.10)
        ax.set_xlabel("$t$"); ax.set_ylabel(r"$\sigma_X^2(t)$")
        ax.set_title(r"$\hat\gamma$ = " + f"{g:.2f}   " + r"$\hat\gamma_{A2}$ = " + f"{g2:.2f}",
                     fontsize=9)
        ax.legend(fontsize=6.5)
fig.suptitle("Analogo della Fig. 2 di Altmann et al. sui quattro corpora\n"
             "lettera: coda rapida e $\\hat\\gamma_{A2}\\simeq1$ — sostantivo: coda larga e "
             "$\\hat\\gamma_{A2}\\simeq\\hat\\gamma$", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "analogo_fig2.png"); plt.show()

In [ ]:
# --- analogo della Fig. 3 di Altmann et al.: diagramma burstiness-correlazione ---
COLTIPO = {"lettera": "#8c8c8c", "spazio": "#8c8c8c", "vocali": "#8c8c8c",
           "funzione": "#0072B2", "appaiata": "#009E73", "keyword": "#D55E00"}
MK = {"lettera": "o", "spazio": "*", "vocali": "X", "funzione": "s",
      "appaiata": "v", "keyword": "D"}
DIM = {"lettera": 26, "spazio": 90, "vocali": 70, "funzione": 34, "appaiata": 38, "keyword": 52}

fig, axes = plt.subplots(1, len(ORDINE), figsize=(4.4*len(ORDINE), 4.5), squeeze=False)
S_ok = A[np.isfinite(A["cv_tau"]) & np.isfinite(A["gamma"])]
for j, c in enumerate(ORDINE):
    ax = axes[0][j]
    sub = S_ok[S_ok.corpus == c]
    for tipo in ["lettera", "spazio", "vocali", "funzione", "appaiata", "keyword"]:
        s = sub[sub.tipo == tipo]
        if not len(s): continue
        ax.scatter(s["cv_tau"], s["gamma"], s=DIM[tipo], marker=MK[tipo], alpha=.7,
                   c=[COLTIPO[tipo]], edgecolors="k", linewidths=.3, label=tipo)
    ax.axhline(1, color="k", lw=1); ax.axvline(1, color="k", lw=1)
    ax.plot([1], [1], marker="*", ms=13, color="k")
    ax.annotate("Poisson", (1, 1), textcoords="offset points", xytext=(7, -13), fontsize=8)
    ax.set_xlabel(r"burstiness  $\sigma_\tau/\langle\tau\rangle$")
    if j == 0:
        ax.set_ylabel(r"correlazione  $\hat\gamma$"); ax.legend(fontsize=7, loc="upper left")
    ax.set_title(ETICHETTE[c], fontsize=10)
xm = np.nanpercentile(S_ok["cv_tau"], 99.5); ym = np.nanpercentile(S_ok["gamma"], 99.5)
for ax in axes[0]:
    ax.set_xlim(0, max(xm, 2)); ax.set_ylim(min(0.85, np.nanpercentile(S_ok["gamma"], 1)),
                                            max(ym, 1.6))
fig.suptitle("Analogo della Fig. 3 di Altmann et al.: diagramma burstiness-correlazione\n"
             f"atteso: lettere in basso a sinistra, sostantivi in alto a destra "
             f"(N = {N_EFF:,} caratteri)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "analogo_fig3.png"); plt.show()

## Shuffling M1 / M2 sui quattro corpora

Altmann et al. generano testi della stessa lunghezza con due manipolazioni: **M1** tiene fisse
le posizioni degli spazi e ricolloca ogni token in un buco della sua misura, distruggendo i
legami verso i livelli sopra le parole; **M2** ricodifica ogni tipo lessicale con una stringa
casuale di pari lunghezza, applicata coerentemente, rimescolando i legami parola-lettera ma
conservando la struttura superiore. La previsione è che **M1 distrugga e M2 preservi** le
correlazioni, e la conferma di M2 è la verifica sperimentale dell'arbitrarietà del segno.

In [ ]:
# --- M1 / M2 sui quattro corpora ---
N_DOC_SH = 10          # documenti per corpus (il calcolo e' ripetuto 3 volte per documento)
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def shuffle_M1(text, r):
    parti = re.split(r"(\s+)", text)
    idx = {}
    for i, p in enumerate(parti):
        if p and not p.isspace():
            idx.setdefault(len(p), []).append(i)
    out = list(parti)
    for L, ii in idx.items():
        vals = [parti[i] for i in ii]
        for i, k in zip(ii, r.permutation(len(vals))):
            out[i] = vals[k]
    return "".join(out)

def shuffle_M2(text, r):
    mappa = {}
    def rep(m):
        w = m.group(0); lw = w.lower()
        if lw not in mappa:
            mappa[lw] = "".join(r.choice(list(LETTERS), size=len(w)))
        return mappa[lw]
    return re.sub(r"[A-Za-z]+", rep, text)

def gamma_lettere(t, N):
    """gamma medio sulle 10 lettere piu' frequenti"""
    tt = t[:N]
    carr = np.array(list(tt.lower()))
    gs = []
    for ch, _ in Counter(x for x in tt.lower() if x.isalpha() and x.isascii()).most_common(10):
        pos = np.flatnonzero(carr == ch)
        if len(pos) < MIN_EVENTS: continue
        g = fit_gamma(transport_sigma2(pos, N), *FIT_RANGE)[0]
        if np.isfinite(g): gs.append(g)
    return float(np.mean(gs)) if gs else np.nan

righe = []
for c in ORDINE:
    docs = documenti_di(c, N_DOC_SH)
    print(f"  {ETICHETTE[c]:17s}", end="", flush=True)
    for lab, testo in docs:
        t = testo[:N_EFF]
        r = np.random.default_rng(seme(c, lab, "M1M2"))
        righe.append(dict(corpus=c, titolo=lab, variante="originale", gamma=gamma_lettere(t, N_EFF)))
        righe.append(dict(corpus=c, titolo=lab, variante="M1",
                          gamma=gamma_lettere(shuffle_M1(t, r), N_EFF)))
        righe.append(dict(corpus=c, titolo=lab, variante="M2",
                          gamma=gamma_lettere(shuffle_M2(t, r), N_EFF)))
        print(".", end="", flush=True)
    print(" ok")
SH = pd.DataFrame(righe)
SH.to_csv(OUT_DIR / "dati" / "shuffling_M1M2.csv", index=False)

print("\ngamma medio sulle lettere\n")
print(f"  {'corpus':17s}{'originale':>12s}{'M1':>12s}{'M2':>12s}{'M1 distrugge?':>16s}")
for c in ORDINE:
    g = SH[SH.corpus == c].groupby("variante")["gamma"].agg(["mean", "std"])
    if not len(g): continue
    o, m1, m2 = g.loc["originale", "mean"], g.loc["M1", "mean"], g.loc["M2", "mean"]
    esito = "si" if (m1 < o - 0.02 and abs(m2 - o) < abs(m1 - o)) else "no"
    print(f"  {ETICHETTE[c]:17s}{o:>7.3f}+-{g.loc['originale','std']:<4.3f}"
          f"{m1:>7.3f}+-{g.loc['M1','std']:<4.3f}{m2:>7.3f}+-{g.loc['M2','std']:<4.3f}"
          f"{esito:>16s}")
print("\n  previsione di Altmann et al.: M1 riporta gamma verso 1, M2 lo conserva.")

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ordine_v = ["originale", "M1", "M2"]
for i, c in enumerate(ORDINE):
    g = SH[SH.corpus == c].groupby("variante")["gamma"].agg(["mean", "std"]).reindex(ordine_v)
    ax.bar(np.arange(3) + .2*(i-1.5), g["mean"], .19, yerr=g["std"].fillna(0), capsize=3,
           label=ETICHETTE[c], color=COLORI[c], alpha=.85)
ax.axhline(1, color="grey", ls=":", lw=1.4)
ax.set_xticks(range(3)); ax.set_xticklabels(ordine_v)
ax.set_ylabel(r"$\hat\gamma$ medio sulle lettere")
ax.set_title("M1 deve distruggere le correlazioni, M2 deve preservarle\n"
             f"({N_DOC_SH} documenti per corpus, N = {N_EFF:,} caratteri)", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "shuffling_M1M2.png"); plt.show()

## Robustezza rispetto alla lunghezza di analisi

Tutti i risultati sono misurati a $\Neff$ fissato. Poiché $\cv$ dipende da $N$, occorre
verificare che le conclusioni non dipendano dal valore scelto. L'analisi è stata ripetuta
integralmente a $N_{\rm eff}=\num{80000}$ caratteri, che è il massimo compatibile con la
lunghezza delle voci di Wikipedia: si perdono 4 terne su 39 (le voci in cui il lato
Grokipedia è più corto).

In [ ]:
# --- confronto fra le due lunghezze di analisi ---
DIR_NEFF = QUI / "confronto_N_eff"
if not (DIR_NEFF / "confronto.json").exists():
    print("Eseguire prima confronto_neff.py; questa cella si limita a leggere i risultati.")
else:
    import json as _json
    R2 = _json.load(open(DIR_NEFF / "confronto.json"))
    a, b = R2["60000"], R2["80000"]
    print(f"terne complete: {a['n_terne']} -> {b['n_terne']}\n")
    print("1. TEST APPAIATO INTERNO (keyword vs controllo)")
    print(f"  {'corpus':17s}{'metrica':9s}{'60k':>9s}{'80k':>9s}{'variaz.':>10s}")
    for c in ORDINE:
        for col in ["cv_tau", "gamma"]:
            x, y = a[f"test_{c}_{col}"][2], b[f"test_{c}_{col}"][2]
            print(f"  {ETICHETTE[c]:17s}{col:9s}{x:>9.2f}{y:>9.2f}{y-x:>+10.2f}")
    print("\n2. DIVARIO GROKIPEDIA - WIKIPEDIA (cv_tau, confronto appaiato)")
    print(f"  {'livello':20s}{'diff 60k':>11s}{'p 60k':>10s}{'diff 80k':>11s}{'p 80k':>10s}")
    for tipo, nome in LIV:
        if f"pair_{tipo}" not in a: continue
        x, y = a[f"pair_{tipo}"], b[f"pair_{tipo}"]
        print(f"  {nome:20s}{x[1]:>+11.3f}{x[3]:>10.1e}{y[1]:>+11.3f}{y[3]:>10.1e}")
    print("\n3. QUOTA DI BURSTINESS q SULLE KEYWORD")
    print(f"  {'corpus':17s}{'60k':>9s}{'80k':>9s}{'variaz.':>10s}")
    for c in ORDINE:
        x, y = a[f"q_{c}_keyword"], b[f"q_{c}_keyword"]
        print(f"  {ETICHETTE[c]:17s}{x:>9.2f}{y:>9.2f}{y-x:>+10.2f}")
    dq60 = a["q_wikipedia_keyword"] - a["q_grok_oggi_keyword"]
    dq80 = b["q_wikipedia_keyword"] - b["q_grok_oggi_keyword"]
    print(f"\n  differenza Wikipedia - Grokipedia:  {dq60:+.2f} a 60k,  {dq80:+.2f} a 80k")
    print("  -> la differenza sulla quota NON e' robusta alla lunghezza della finestra:")
    print("     va riportata come non conclusiva.")
    print("\n4. CODA DELLE KEYWORD")
    print(f"  {'corpus':17s}{'mu 60k':>9s}{'mu 80k':>9s}{'tau_c 60k':>11s}{'tau_c 80k':>11s}")
    for c in ORDINE:
        if f"mu_{c}" not in a: continue
        print(f"  {ETICHETTE[c]:17s}{a[f'mu_{c}']:>9.2f}{b[f'mu_{c}']:>9.2f}"
              f"{a[f'tauc_{c}']:>11.1f}{b[f'tauc_{c}']:>11.1f}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    xs = np.arange(len(LIV))
    ax = axes[0]
    for i, c in enumerate(ORDINE):
        v60 = [a[f"cv_tau_{c}_{t}"] for t, _ in LIV]
        v80 = [b[f"cv_tau_{c}_{t}"] for t, _ in LIV]
        ax.plot(xs, v60, "-o", color=COLORI[c], ms=6, lw=1.7, label=ETICHETTE[c])
        ax.plot(xs, v80, "--s", color=COLORI[c], ms=5, lw=1.2, alpha=.6)
    ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
    ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
    ax.set_title("A) burstiness: 60k (pieno) e 80k (tratteggio)", fontsize=10)
    ax.legend(fontsize=7)
    ax = axes[1]
    d60 = [a[f"pair_{t}"][1] for t, _ in LIV]
    d80 = [b[f"pair_{t}"][1] for t, _ in LIV]
    ax.plot(xs, d60, "-o", color="#0072B2", ms=7, lw=1.8, label="60k")
    ax.plot(xs, d80, "--s", color="#D55E00", ms=6, lw=1.6, label="80k")
    ax.axhline(0, color="grey", ls=":", lw=1.2)
    ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
    ax.set_ylabel("Grokipedia − Wikipedia")
    ax.set_title("B) divario appaiato, invariato", fontsize=10); ax.legend(fontsize=8)
    ax = axes[2]
    q60 = [a[f"q_{c}_keyword"] for c in ORDINE]
    q80 = [b[f"q_{c}_keyword"] for c in ORDINE]
    xs2 = np.arange(len(ORDINE))
    ax.bar(xs2 - .19, q60, .36, label="60k", color="#0072B2", alpha=.85)
    ax.bar(xs2 + .19, q80, .36, label="80k", color="#D55E00", alpha=.85)
    ax.set_xticks(xs2)
    ax.set_xticklabels([ETICHETTE[c].replace(" ", "\n") for c in ORDINE], fontsize=7.5)
    ax.set_ylabel("quota di burstiness $q$ (keyword)")
    ax.set_title("C) la quota NON e' robusta:\nil divario Wikipedia-Grokipedia si annulla",
                 fontsize=10)
    ax.legend(fontsize=8)
    fig.suptitle("Robustezza rispetto alla lunghezza di analisi", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure" / "robustezza_neff.png"); plt.show()

In [ ]:
L = []
L.append("ALTMANN SU TRE CORPORA APPAIATI PER TITOLO")
L.append("=" * 68)
L.append(f"terne complete: {len(USABILI)} | esclusi: {ESCLUSI} | N_EFF = {N_EFF:,} caratteri")
L.append(f"riferimento letterario: {len(SEG)} segmenti di Guerra e pace alla stessa lunghezza")
L.append("")
L.append("1) L'EFFETTO DI ALTMANN DENTRO CIASCUN CORPUS (keyword vs controllo appaiato)")
for met in ["cv_tau", "gamma"]:
    L.append(f"   {met}:")
    for c in ORDINE:
        x = PT[(PT.corpus == c) & (PT.metrica == met)].iloc[0]
        L.append(f"     {ETICHETTE[c]:18s} {int(x['vittorie']):4d}/{int(x['confronti']):4d} = "
                 f"{x['frazione']:.2f} [{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]  p={x['p']:.1e}"
                 f"   {'effetto presente' if x['ic_lo'] > 0.5 else 'NON dimostrato'}")
L.append("")
L.append("2) CONFRONTI APPAIATI (Wilcoxon sulle differenze per titolo)")
for c1, c2 in COPPIE:
    for tipo in ["keyword", "lettera"]:
        for col in ["cv_tau", "gamma"]:
            x = CONF[(CONF.corpus_1 == c1) & (CONF.corpus_2 == c2) &
                     (CONF.tipo == tipo) & (CONF.metrica == col)].iloc[0]
            if not np.isfinite(x["p"]): continue
            segno = "significativa" if x["p"] < 0.05 else "non significativa"
            L.append(f"   {ETICHETTE[c1]} vs {ETICHETTE[c2]}, {tipo}/{col}: "
                     f"diff mediana {x['mediana_diff']:+.3f}, p={x['p']:.1e} ({segno})")
L.append("")
L.append("3) EFFETTO DELLA REVISIONE (v0.1 -> oggi)")
if len(REV):
    for g in REV["gruppo"].unique():
        s = REV[(REV.gruppo == g) & (REV.tipo == "keyword") & (REV.metrica == "cv_tau")]
        if len(s):
            x = s.iloc[0]
            L.append(f"   {g} ({int(x['n'])} articoli): burstiness keyword "
                     f"{x['v01']:.2f} -> {x['oggi']:.2f} (Wikipedia {x['wiki']:.2f}); "
                     f"la revisione {'avvicina' if x['avvicina'] else 'allontana'}")
    n_avv = int(REV["avvicina"].sum()); n_tot = len(REV)
    L.append(f"   complessivamente: la revisione avvicina a Wikipedia in {n_avv}/{n_tot} "
             f"combinazioni (gruppo x livello x metrica)")
L.append("")
L.append("AVVERTENZE")
L.append("  - Grokipedia ha frasi molto piu' lunghe di Wikipedia (4.38 vs 6.92 frasi ogni")
L.append("    1000 caratteri). Il tempo qui si conta in caratteri, quindi la scala dei tau")
L.append("    non e' la stessa nei due corpora: e' un confonditore per ogni differenza.")
L.append("  - Il riferimento letterario non e' appaiato per argomento: si confronta solo")
L.append("    come livello medio, non con test appaiati.")
L.append(f"  - sigma_tau/<tau> dipende da N: tutti i valori qui sono a N = {N_EFF:,} e non")
L.append("    sono confrontabili con quelli del paper, misurati su libri interi.")
L.append("")
L.append("4) DECOMPOSIZIONE DELL'EQ. 4  q = (gamma_A2 - 1)/(gamma - 1)")
for c_ in ORDINE:
    l_ = Q[(Q.corpus == c_) & (Q.tipo == "lettera")]["q"]
    k_ = Q[(Q.corpus == c_) & (Q.tipo == "keyword")]["q"]
    if len(l_) and len(k_) and np.isfinite(l_.iloc[0]):
        L.append(f"   {ETICHETTE[c_]:17s} lettere q={l_.iloc[0]:+.2f}   keyword q={k_.iloc[0]:+.2f}")
L.append("   -> la dissociazione della Fig. 2 di Altmann e' presente in tutti i corpora:")
L.append("      lettere correlate ma non bursty, keyword bursty.")
L.append(f"   gamma > 1 significativo in {int((LR['p_vs_1'] < 0.05).sum())}/{len(LR)} celle,"
         f" gamma_A1 medio {LR['gamma_A1'].mean():.3f}")
S = "\n".join(L)
(OUT_DIR / "sintesi.txt").write_text(S, encoding="utf-8")
print(S)